# Product Review Sentiment Analyzer — EDA

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

os.makedirs('plots', exist_ok=True)

df = pd.read_csv('../data/Reviews.csv')
print('Shape:', df.shape)

In [ ]:
print('Dtypes:\n', df.dtypes)
print('\nNull values:\n', df.isnull().sum())

In [ ]:
print('Sample rows:')
df.head(3)

## Star Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
score_counts = df['Score'].value_counts().sort_index()
sns.barplot(x=score_counts.index, y=score_counts.values, palette='Blues_d', ax=ax)
ax.set_title('Distribution of Star Ratings', fontsize=14)
ax.set_xlabel('Star Rating')
ax.set_ylabel('Count')
for i, v in enumerate(score_counts.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('plots/rating_distribution.png', dpi=150)
plt.show()
print('Saved: plots/rating_distribution.png')

## Derive Sentiment Labels

In [ ]:
def score_to_sentiment(score):
    if score in [1, 2]:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(score_to_sentiment)
print(df['sentiment'].value_counts())

## Sentiment Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ['positive', 'neutral', 'negative']
palette = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
counts = df['sentiment'].value_counts()
bars = [counts[s] for s in order]
colors = [palette[s] for s in order]
ax.bar(order, bars, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Sentiment Label Distribution', fontsize=14)
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
for i, v in enumerate(bars):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('plots/sentiment_distribution.png', dpi=150)
plt.show()
print('Saved: plots/sentiment_distribution.png')

## Word Clouds

In [ ]:
df = df.dropna(subset=['Text'])

pos_text = ' '.join(df[df['sentiment'] == 'positive']['Text'].astype(str).tolist())
neg_text = ' '.join(df[df['sentiment'] == 'negative']['Text'].astype(str).tolist())

wc_pos = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=200).generate(pos_text)
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc_pos, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud — Positive Reviews', fontsize=14)
plt.tight_layout()
plt.savefig('plots/wordcloud_positive.png', dpi=150)
plt.show()
print('Saved: plots/wordcloud_positive.png')

wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=200).generate(neg_text)
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc_neg, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud — Negative Reviews', fontsize=14)
plt.tight_layout()
plt.savefig('plots/wordcloud_negative.png', dpi=150)
plt.show()
print('Saved: plots/wordcloud_negative.png')

## Review Length Analysis

In [ ]:
df['review_length'] = df['Text'].astype(str).apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(10, 5))
for sentiment, color in [('positive', '#2ecc71'), ('neutral', '#95a5a6'), ('negative', '#e74c3c')]:
    subset = df[df['sentiment'] == sentiment]['review_length']
    subset_clipped = subset[subset <= 300]
    ax.hist(subset_clipped, bins=50, alpha=0.6, color=color, label=sentiment)
ax.set_title('Review Length Distribution by Sentiment', fontsize=14)
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('plots/review_length_distribution.png', dpi=150)
plt.show()
print('Saved: plots/review_length_distribution.png')